# 03 — LangGraph Pipeline

Defines:
- `schemas.py` — the Pydantic `AskRequest` / `AskResponse` models (the JSON output
  schema: `answer`, `sources`, `confidence`).
- `graph.py` — the `StateGraph` with a `TypedDict` state and 3 nodes
  (`classify_intent`, `retrieve_and_answer`, `direct_answer`), wired with a
  conditional edge, plus the `MOCK_LLM` branching required in every generation step.

`MOCK_LLM` (env var) controls the *only* optional/ungraded part of this pipeline:
- unset or `"1"` (**default, graded**): rule-based classification + canned templated
  answers, zero LLM calls, fully deterministic.
- `"0"` (**optional extension**): calls a real LLM (e.g. Groq free tier) for
  classification and generation, with schema-validation retries.


In [1]:
%%writefile schemas.py
"""
Pydantic schemas for the /ask endpoint.

AskResponse is the JSON output schema enforced on every final answer:
  - answer: str
  - sources: list[str]   (chunk/doc ids used; empty for general_question)
  - confidence: float    (0.0-1.0)
"""
from typing import List
from pydantic import BaseModel, Field


class AskRequest(BaseModel):
    query: str = Field(..., min_length=1, description="The customer's question.")


class AskResponse(BaseModel):
    answer: str
    sources: List[str] = Field(default_factory=list)
    confidence: float = Field(..., ge=0.0, le=1.0)


Writing schemas.py


In [2]:
%%writefile graph.py
"""
LangGraph-orchestrated RAG pipeline for the Zepto support assistant.

Stages (see README for the full architecture write-up):
  classify_intent      -> routes each query (policy_question | general_question)
  retrieve_and_answer   -> real ChromaDB retrieval (always) + generation (MOCK_LLM-gated)
  direct_answer         -> generation only (MOCK_LLM-gated), no retrieval

MOCK_LLM env var:
  unset or "1" -> graded baseline: no LLM calls anywhere in the graph.
  "0"          -> optional extension: real LLM calls for classification/generation,
                  via Groq (or any free-tier-compatible OpenAI-style client).
"""
import os
import json
from typing import TypedDict, List, Optional, Literal

from langgraph.graph import StateGraph, END

from ingest import retrieve_top_k
from prompt_template import build_messages
from schemas import AskResponse

MOCK_LLM = os.environ.get("MOCK_LLM", "1") != "0"  # True = mock (default/graded)

POLICY_KEYWORDS = [
    "delivery", "return", "refund", "membership",
    "tracking", "cancel", "gift card", "support hours",
]

CANNED_GENERAL_ANSWER = "I can only answer questions about Zepto policies right now."


# ---------------------------------------------------------------------------
# Graph state
# ---------------------------------------------------------------------------
class GraphState(TypedDict, total=False):
    query: str
    intent: Literal["policy_question", "general_question"]
    retrieved_chunks: List[dict]
    answer: str
    sources: List[str]
    confidence: float


# ---------------------------------------------------------------------------
# Optional real-LLM client (only constructed when MOCK_LLM=0)
# ---------------------------------------------------------------------------
def _get_llm_client():
    """
    Lazily build an OpenAI-compatible client pointed at Groq's free-tier API.
    Only ever called from the MOCK_LLM=0 branches below.
    """
    from openai import OpenAI  # local import: not a required dependency in mock mode
    api_key = os.environ.get("GROQ_API_KEY")
    if not api_key:
        raise RuntimeError(
            "MOCK_LLM=0 requires GROQ_API_KEY to be set (optional extension only; "
            "the graded baseline runs with MOCK_LLM left at its default and never "
            "reaches this code path)."
        )
    return OpenAI(api_key=api_key, base_url="https://api.groq.com/openai/v1")


def _call_llm_for_json(messages, model="llama-3.1-8b-instant", max_retries=2):
    """
    Call the real LLM and validate its output against AskResponse, retrying up to
    `max_retries` additional times with a corrective instruction on validation
    failure. Only reached when MOCK_LLM=0. Never called in the graded baseline.
    """
    client = _get_llm_client()
    attempt_messages = list(messages)
    last_error = None

    for attempt in range(max_retries + 1):
        resp = client.chat.completions.create(
            model=model,
            messages=attempt_messages,
            temperature=0.0,
        )
        raw = resp.choices[0].message.content.strip()
        try:
            data = json.loads(raw)
            validated = AskResponse(**data)
            return validated
        except Exception as e:  # JSON decode error or Pydantic ValidationError
            last_error = e
            attempt_messages = list(messages) + [
                {"role": "assistant", "content": raw},
                {"role": "user", "content": (
                    "That response was not valid JSON matching the required schema "
                    f"(answer: str, sources: list[str], confidence: float 0-1). "
                    f"Validation error: {e}. Reply again with ONLY a corrected JSON "
                    "object and nothing else."
                )},
            ]

    # Exhausted retries: clearly-marked error response, not a silent failure.
    return AskResponse(
        answer=f"[ERROR] LLM failed to produce a schema-valid response after "
               f"{max_retries} retries. Last error: {last_error}",
        sources=[],
        confidence=0.0,
    )


# ---------------------------------------------------------------------------
# Node 1: classify_intent
# ---------------------------------------------------------------------------
def classify_intent(state: GraphState) -> GraphState:
    query = state["query"]

    if MOCK_LLM:
        # Mock mode (graded baseline): keyword heuristic, no LLM call.
        lowered = query.lower()
        intent = "policy_question" if any(kw in lowered for kw in POLICY_KEYWORDS) else "general_question"
    else:
        # Optional extension: ask the real LLM to classify.
        client = _get_llm_client()
        resp = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[
                {"role": "system", "content": (
                    "Classify the user's question as exactly one word: "
                    "'policy_question' if it concerns Zepto's delivery, returns, "
                    "membership, tracking, cancellation, damaged/missing items, gift "
                    "cards, or support hours policies, otherwise 'general_question'. "
                    "Reply with only that one word."
                )},
                {"role": "user", "content": query},
            ],
            temperature=0.0,
        )
        raw = resp.choices[0].message.content.strip().lower()
        intent = "policy_question" if "policy_question" in raw else "general_question"

    return {**state, "intent": intent}


# ---------------------------------------------------------------------------
# Node 2: retrieve_and_answer  (policy_question path)
# ---------------------------------------------------------------------------
def retrieve_and_answer(state: GraphState) -> GraphState:
    query = state["query"]

    # Retrieval always runs for real, in both modes (no API key/network needed).
    chunks = retrieve_top_k(query, k=3)
    top_chunk = chunks[0] if chunks else None

    if MOCK_LLM:
        # Mock mode (graded baseline): canned templated answer, no LLM call.
        if top_chunk:
            snippet = top_chunk["text"][:200]
            answer = f"Based on the retrieved context: {snippet}"
            sources = [c["chunk_id"] for c in chunks]
        else:
            answer = "Based on the retrieved context: no relevant policy information was found."
            sources = []
        confidence = 1.0
    else:
        # Optional extension: real LLM, grounded only in retrieved chunks.
        messages = build_messages(query, chunks)
        validated = _call_llm_for_json(messages)
        answer = validated.answer
        sources = validated.sources
        confidence = validated.confidence

    return {
        **state,
        "retrieved_chunks": chunks,
        "answer": answer,
        "sources": sources,
        "confidence": confidence,
    }


# ---------------------------------------------------------------------------
# Node 3: direct_answer  (general_question path, no retrieval)
# ---------------------------------------------------------------------------
def direct_answer(state: GraphState) -> GraphState:
    query = state["query"]

    if MOCK_LLM:
        # Mock mode (graded baseline): fixed canned string, no LLM call.
        answer = CANNED_GENERAL_ANSWER
        confidence = 1.0
    else:
        # Optional extension: prompt the LLM directly, no retrieval context.
        client = _get_llm_client()
        resp = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[
                {"role": "system", "content": (
                    "You are a general assistant. The user's question is unrelated to "
                    "Zepto policies. Answer briefly and helpfully in 1-2 sentences."
                )},
                {"role": "user", "content": query},
            ],
            temperature=0.3,
        )
        answer = resp.choices[0].message.content.strip()
        confidence = 0.7  # heuristic: no grounding/schema-validated confidence available here

    return {**state, "answer": answer, "sources": [], "confidence": confidence}


# ---------------------------------------------------------------------------
# Conditional routing (does not depend on MOCK_LLM)
# ---------------------------------------------------------------------------
def route_by_intent(state: GraphState) -> str:
    return "retrieve_and_answer" if state["intent"] == "policy_question" else "direct_answer"


# ---------------------------------------------------------------------------
# Build the graph
# ---------------------------------------------------------------------------
def build_graph():
    graph = StateGraph(GraphState)

    graph.add_node("classify_intent", classify_intent)
    graph.add_node("retrieve_and_answer", retrieve_and_answer)
    graph.add_node("direct_answer", direct_answer)

    graph.set_entry_point("classify_intent")
    graph.add_conditional_edges(
        "classify_intent",
        route_by_intent,
        {
            "retrieve_and_answer": "retrieve_and_answer",
            "direct_answer": "direct_answer",
        },
    )
    graph.add_edge("retrieve_and_answer", END)
    graph.add_edge("direct_answer", END)

    return graph.compile()


_compiled_graph = None


def get_graph():
    global _compiled_graph
    if _compiled_graph is None:
        _compiled_graph = build_graph()
    return _compiled_graph


def ask(query: str) -> AskResponse:
    """Run the full graph for one query and return a validated AskResponse."""
    graph = get_graph()
    result = graph.invoke({"query": query})
    return AskResponse(
        answer=result["answer"],
        sources=result.get("sources", []),
        confidence=result["confidence"],
    )


if __name__ == "__main__":
    for q in ["How long do I have to return a spoiled item?", "What's the capital of France?"]:
        r = ask(q)
        print(q, "->", r.model_dump())


Writing graph.py


In [4]:
!pip3 install langgraph

     ---------------------------------------- 0.0/248.9 kB ? eta -:--:--
     - -------------------------------------- 10.2/248.9 kB ? eta -:--:--
     ---------- -------------------------- 71.7/248.9 kB 991.0 kB/s eta 0:00:01
     ---------- -------------------------- 71.7/248.9 kB 991.0 kB/s eta 0:00:01
     -------------- --------------------- 102.4/248.9 kB 590.8 kB/s eta 0:00:01
     ----------------- ------------------ 122.9/248.9 kB 554.9 kB/s eta 0:00:01
     ---------------------- ------------- 153.6/248.9 kB 573.4 kB/s eta 0:00:01
     ------------------------- ---------- 174.1/248.9 kB 655.4 kB/s eta 0:00:01
     ----------------------------- ------ 204.8/248.9 kB 567.2 kB/s eta 0:00:01
     -------------------------------- --- 225.3/248.9 kB 551.4 kB/s eta 0:00:01
     ---------------------------------- - 235.5/248.9 kB 515.5 kB/s eta 0:00:01
     ------------------------------------ 248.9/248.9 kB 509.9 kB/s eta 0:00:00
     ---------------------------------------- 0.0/41.


[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
# Sanity-check the graph structure and both routing branches, in mock mode.
import os
os.environ.setdefault("MOCK_LLM", "1")

from graph import ask, MOCK_LLM, POLICY_KEYWORDS
print("MOCK_LLM (mock mode active):", MOCK_LLM)

policy_result = ask("What is your refund policy for damaged items?")
print("\n[policy_question example]")
print(policy_result.model_dump())

general_result = ask("What's a good recipe for banana bread?")
print("\n[general_question example]")
print(general_result.model_dump())

assert policy_result.sources, "policy_question answers should cite sources"
assert general_result.sources == [], "general_question answers should have empty sources"
print("\nBoth graph branches (retrieve_and_answer, direct_answer) verified.")


c:\Users\Srivatsav\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MOCK_LLM (mock mode active): True


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7358.55it/s]



[policy_question example]
{'answer': "Based on the retrieved context: If an order arrives with damaged, spoiled, or missing items, customers must report it within 24 hours of delivery through the 'Report an Issue' button on the order page. Zepto ships a free replacement", 'sources': ['doc_06_c0', 'doc_02_c0', 'doc_02_c1'], 'confidence': 1.0}

[general_question example]
{'answer': 'I can only answer questions about Zepto policies right now.', 'sources': [], 'confidence': 1.0}

Both graph branches (retrieve_and_answer, direct_answer) verified.
